In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point

Part 1: Station spatial data preprocessing

In [2]:
lake_names_std = ['Zhari Namco', 'DawaCo', 'DazegCo', 'BamCo', 'PengCo',
                  'NamCo', 'SilingCo', 'MemarCo', 'Luotuo lake', 'LumajiangdongCo',
                  'Jieze Caka', 'Kalakuli']

lake_coords = [(85.61, 30.93), (84.96, 31.24), (87.53, 31.89), (90.58, 31.26), (90.97, 31.50),
               (90.60, 30.74), (88.63, 30.95), (82.31, 34.22), (81.86, 34.46), (81.61, 34.02),
               (80.90, 33.95), (75.03, 38.43)]

df_stations = pd.DataFrame({
    'station_id': range(len(lake_names_std)),
    'lake_name': lake_names_std,
    'lon': [xy[0] for xy in lake_coords],
    'lat': [xy[1] for xy in lake_coords]
})

Part 2: Read and clean water level data

In [3]:
# Data 1: lei(2018)_tibet_lakes
df_1 = pd.read_excel('data/validation/lake-height/lei2018_tibet_lakes/tibet_lakes.xlsx', usecols=['sno', 'date', 'waterLevel'])
df_1.columns = ['lake_name', 'time', 'water_level']

# Data 2: lei2022_tibet_lakes
lake_columns_config = {
    'Lumajiangdong_Co': {'date_col': 'date', 'water_level_col': 'LMJD_water_level (m)'},
    'Memar_Co': {'date_col': 'date', 'water_level_col': 'Memar Co_water_level (m)'},
    'Luotuo_lake': {'date_col': 'date', 'water_level_col': 'Luotuo lake_water_level'},
    'Jieze_Caka': {'date_col': 'date', 'water_level_col': 'Jieze Caka_water level'}
}
df_2_raw = pd.read_excel('data/validation/lake-height/lei2022_tibet_lakes/lei2022_tibet_lakes.xlsx')
df_2_list = []
for name, cols in lake_columns_config.items():
    temp = df_2_raw[list(cols.values())].dropna().copy()
    temp.columns = ['time', 'water_level']
    temp['lake_name'] = name
    df_2_list.append(temp)
df_2 = pd.concat(df_2_list, ignore_index=True)

# Data 3: wang2018_namco
df_3 = pd.read_excel('data/validation/lake-height/wang2018_namco/wang2018_namco.xls', usecols=['date', 'water_level'])
df_3.columns = ['time', 'water_level']
df_3['lake_name'] = 'Namco'
df_3['water_level'] /= 100 

# Data 4: xie2021_kalakuli_lake
df_4_list = []
for year in range(2011, 2020):
    try:
        temp = pd.read_excel('data/validation/lake-height/xie2021_kalakuli_lake/卡拉库里湖水位_2011_2019.xlsx', sheet_name=str(year), usecols=['时间', '水位（cm）'])
        temp.columns = ['time', 'water_level']
        temp['water_level'] /= 100
        temp['lake_name'] = 'kalakuli'
        df_4_list.append(temp)
    except Exception:
        pass
df_4 = pd.concat(df_4_list, ignore_index=True)

# Data 5: zhang2018_silingco
df_5_raw = pd.read_excel("data/validation/lake-height/zhang2018_silingco/zhang2018_silingco.xls", sheet_name='气象数据集', skiprows=2).iloc[:, 1:4]
df_5_raw.columns = ['time', 'pressure', 'temp']
df_5_raw['time'] = df_5_raw['time'].apply(lambda x: pd.to_datetime(str(x).strip().replace('上午', 'AM').replace('下午', 'PM'), format='%m/%d/%y %p%I时%M分%S秒', errors='coerce'))
df_5_raw = df_5_raw.dropna(subset=['time', 'pressure', 'temp'])

# Calculate water depth (altitude=4551)
P_atm = 101325 * (1 - 0.0065 * 4551 / 288.15) ** (9.80665 / (287.05 * 0.0065))
df_5_raw['water_level'] = (df_5_raw['pressure'] * 1000 - P_atm) / ((1000 * (1 - (df_5_raw['temp'] + 288.9414) * (df_5_raw['temp'] - 3.9863)**2 / (508929.2 * (df_5_raw['temp'] + 68.12963)))) * 9.81)
df_5 = df_5_raw[['time', 'water_level']].copy()
df_5['lake_name'] = 'silingco'

d:\miniconda3\envs\courses\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Part 3: Merge data and export to NetCDF

In [ ]:
df_all = pd.concat([df_1, df_2, df_3, df_4, df_5], ignore_index=True)
df_all['time'] = pd.to_datetime(df_all['time'])

name_mapping = {
    'zhariNamco': 'Zhari Namco', 'Lumajiangdong_Co': 'LumajiangdongCo',
    'Memar_Co': 'MemarCo', 'Luotuo_lake': 'Luotuo lake', 'Jieze_Caka': 'Jieze Caka',
    'Namco': 'NamCo', 'kalakuli': 'Kalakuli', 'silingco': 'SilingCo'
}
df_all['lake_name'] = df_all['lake_name'].replace(name_mapping)

# Assign station_index by unified lake name
name_to_id = dict(zip(df_stations['lake_name'], df_stations['station_id']))
df_all['station_index'] = df_all['lake_name'].map(name_to_id)
df_all = df_all.dropna(subset=['water_level', 'station_index']) # Drop NaN values

# station
ds_station = xr.Dataset(
    {
        "name": (["station"], df_stations['lake_name'].values),
        "lon": (["station"], df_stations['lon'].values),
        "lat": (["station"], df_stations['lat'].values),
    },
    coords={"station_id": (["station"], df_stations['station_id'].values)}
)

# observation
ds_obs = xr.Dataset(
    {
        "station_index": (["observation"], df_all['station_index'].astype(int).values),
        "time": (["observation"], df_all['time'].values),
        "water_level": (["observation"], df_all['water_level'].values),
    }
)

# output
ds_final = xr.merge([ds_station, ds_obs])

output_path = 'data/validation/lake_height/lake_station_lakes_all.nc'
ds_final.to_netcdf(output_path)